# KNative Messaging Library Demo

This notebook demonstrates the Python messaging library for CloudEvents over KNative Eventing.

In [ ]:
import sys
sys.path.insert(0, '..')

from messaging import CloudEvent, Disposition, MessageContext, PublishOptions
from messaging.knative import KNativeEventingPublisher

print('Library imported successfully!')

## 1. Create and inspect a CloudEvent

In [ ]:
event = CloudEvent(
    type='com.example.order.created',
    source='/orders/service',
    data={'order_id': 12345, 'total': 99.99, 'currency': 'USD'},
    subject='orders',
)

print(f'Event ID:   {event.id}')
print(f'Type:       {event.type}')
print(f'Source:     {event.source}')
print(f'Time:       {event.time}')
print(f'Data:       {event.data}')
print()
print('HTTP headers (binary content mode):')
for k, v in event.to_headers().items():
    print(f'  {k}: {v}')

## 2. Publish an event to the demo backend

In [ ]:
# Point the publisher at our demo backend instead of a real KNative broker
publisher = KNativeEventingPublisher(broker_url='http://localhost:8000/api/send')

import httpx, json

# For the demo we POST directly to the REST API since mock mode loops back internally.
async def send_demo_event(event: CloudEvent) -> None:
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            'http://localhost:8000/api/send',
            json={
                'type': event.type,
                'source': event.source,
                'data': event.data,
                'subject': event.subject,
            },
        )
        print(f'Status: {resp.status_code}')
        print(f'Response: {resp.json()}')

await send_demo_event(event)

## 3. Publish with custom options

In [ ]:
options = PublishOptions(
    timeout=10.0,
    headers={'X-Correlation-Id': 'notebook-demo-001'},
)
print(f'Timeout: {options.timeout}s')
print(f'Extra headers: {options.headers}')

# Send another event
event2 = CloudEvent(
    type='com.example.notification',
    source='/notebook',
    data={'text': 'Published with custom options!', 'priority': 'high'},
)
await send_demo_event(event2)

## 4. Define a message handler

In [ ]:
from messaging import MessageHandler

class OrderHandler:
    """Processes order events."""

    async def handle(self, event: CloudEvent, context: MessageContext) -> Disposition:
        print(f'Received order event: {event.data}')
        print(f'Message ID: {context.message_id}, Delivery: {context.delivery_count}')
        # Process the order...
        return Disposition.COMPLETE

handler = OrderHandler()

# Verify it satisfies the Protocol
assert isinstance(handler, MessageHandler)
print('OrderHandler satisfies MessageHandler protocol ✓')

# Test it locally
ctx = MessageContext(message_id=event.id, delivery_count=1)
result = await handler.handle(event, ctx)
print(f'Disposition: {result.value}')